# Practica 04: Analisis de Datos Exploratorios con Python y jupiter Notebook en un Dataset de productos de Amazon Store

**Programa de Estudios:** Ingenieria en Entornos Virtuales y Negocios Digitales \
**Asignatura:** Analitica de Datos para Negocios Digitales  \
**Docente:** M.T.I Marco A. Ramirez Hernandez \
**Periodo:** Mayo - Agosto 2026

### Unidad 2: Preparacion de los Datos 

**Nombre:** Karol Janeeth Gonzalez Vargas\
**Matricula:** 230323  \
**Grado y grupo:** 9°A IEVND 

<div style="background: linear-gradient(135deg, #288BC7 0%, #232F3E 100%);
padding: 30px; border-radius: 15px; text-align: center;
margin-bottom: 20px;">

<h1 style="color: white; font-size: 2.2em; margin: 0;">
🛒 Productos de Amazon Store – Análisis Exploratorio de Datos Completo & Aprendizaje Automático Predictivo (ML)
</h1>

<p style="color: #FFD700; font-size: 1.1em; margin-top: 10px;">
Predicción de Precios · Análisis de Categorías · Ingeniería de Funcionalidades · Comparación de Modelos
</p>

</div>

---

## 📋 Tabla de Contenidos 
| # | Seccion | Descripcion |
|---|---|---|
| 1 | [Instalaciones & Carga de Datos](#s1) | Librerías, carga del CSV, primeros comandos de estructura |
| 2 | [Diccionario de Datos](#s2) | Explicaciondel contenido de columnas y calcular que 1% de datos faltantes |

## 1. Instalaciones & Carga de Datos 🔧 <a id='s1'></a> 

<div style="background:#f0f8ff; padding:12px; border-left:4px solid #FF9900; border-radius:5px 0 0; color:#0066cc;">

<b>Dataset:</b> Amazon India Listas de Productos – 1,436 productos entre Libros, Kindle, Deportes & más<br>

<b>Objetivo:</b> Predecir el precio del producto (INR) desde la categoría, longitud del nombre, y disponibilidad<br>

<b>DataSource:</b> Web-scraped de páginas de productos Amazon.in

</div>

In [2]:
# =============================================
# SECTION 1 - Setup & Data  Loading
# =============================================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as ptl
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

print(f'✅ Pandas : {pd.__version__}')
print(f'✅ NumPy : {np.__version__}')
print('✅ Todas las librerías cargaron con éxito!')


✅ Pandas : 2.3.3
✅ NumPy : 2.3.5
✅ Todas las librerías cargaron con éxito!


In [3]:
# Load dataset
df_raw = pd.read_csv('./amazon_products_data.csv', encoding='latin-1')

print(f'📦 Shape: {df_raw.shape}')
print(f'📋 Columns: {df_raw.columns.tolist()}')

df_raw.head(3)

📦 Shape: (1436, 17)
📋 Columns: ['rl', 'asin', 'name', 'overview', 'price', 'currency', 'availability', 'brand', 'about_item', 'img_source', 'description', 'specifications', 'primary_category', 'category_1', 'category_2', 'category_3', 'breadcrumbs']


,rl,asin,name,overview,price,currency,availability,brand,about_item,img_source,description,specifications,primary_category,category_1,category_2,category_3,breadcrumbs
0,https://www.amazon.in/dp/1788832566,1788832566,Hands-On Design Patterns with C++: Solve commo...,[],"1,600.00",INR,NaN,by \r\n Fedor G. Pikus (...,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,[],Books,Computers & Internet,Programming & Software Development,NaN,Books | Computers & Internet | Programming & S...
1,https://www.amazon.in/dp/0766199592,0766199592,The Blazing Horizon The True Story of Pawnee B...,[],"2,399.00",INR,Usually dispatched in 13 to 14 days.,by \r\nErnest Lynn \r\n(Author),NaN,https://images-na.ssl-images-amazon.com/images...,NaN,[],Books,Action & Adventure,NaN,NaN,Books | Action & Adventure
2,https://www.amazon.in/dp/B071ZCD9XH,B071ZCD9XH,Fast and easy lunch box: A collection of 15eas...,[],NaN,NaN,NaN,by \r\nNinety Nine Cents Pr...,NaN,https://m.media-amazon.com/images/I/41g9Z21QuG...,NaN,[],Kindle Store,Kindle eBooks,"Crafts, Home & Lifestyle",NaN,"Kindle Store | Kindle eBooks | Crafts, Home & ..."


## 2. Diccionario de Datos del Data Frame

In [4]:
# Colum dictionary 
col_info = {
    'rl'               :'URL del producto en Amazon.in',
    'asin'             :'Numero Estandarizado de Indentificacion por Amazon(unique ID)',
    'name'             :'Nombre o Titulo del Producto',
    'overview'         :'Breve reseña del Producto(mayoria vacio [])',
    'price'            :'Precio como string (e.g "1,600.00")',
    'currency'         :'Codigo de Divisa(Todas en INR)',
    'availability'     :'Texto del Satatus del stock(Inventario)',
    'brand'            :'Nombre de la Marca/Autor',
    'about_item'       :'Detalles/Especificaciones(mayoria faltante)',
    'img_source'       :'URLs de Imagenes del producto',
    'description'      :'Descripcion detallada y completa(mayoria faltante)',
    'specifications'   :'Especificaciones Tecnicas',
    'primary_category' :'Categoria Principal(Libros, Kindle, Deportes...)',
    'category_1'       :'Subcategoria de Nivel 1',
    'category_2'       :'Subcategoria de Nivel 2)',
    'category_3'       :'Subcategoria de Nivel 3',
    'breadcrumbs'      :'Ruta de Categorias',
}
print('📖 Diccionario por columnas:')

for col, desc in col_info.items():
    missing_pct = df_raw[col].isnull().mean() * 100
    print(
        f' {col:20s} |' 
        f' {desc[:45]:45s} |'
        f' Missing: {missing_pct:.0f}%'
    )


📖 Diccionario por columnas:
 rl                   | URL del producto en Amazon.in                 | Missing: 0%
 asin                 | Numero Estandarizado de Indentificacion por A | Missing: 0%
 name                 | Nombre o Titulo del Producto                  | Missing: 0%
 overview             | Breve reseña del Producto(mayoria vacio [])   | Missing: 0%
 price                | Precio como string (e.g "1,600.00")           | Missing: 24%
 currency             | Codigo de Divisa(Todas en INR)                | Missing: 24%
 availability         | Texto del Satatus del stock(Inventario)       | Missing: 33%
 brand                | Nombre de la Marca/Autor                      | Missing: 1%
 about_item           | Detalles/Especificaciones(mayoria faltante)   | Missing: 92%
 img_source           | URLs de Imagenes del producto                 | Missing: 0%
 description          | Descripcion detallada y completa(mayoria falt | Missing: 92%
 specifications       | Especificaciones Te

## 3.Limpieza de Datos

<div style="background:#fff8e1; padding:12px; border-left:4px solid #FF9900; border-radius:5px; color: #0066cc;">
<b>Pasos clave para la limpieza:</b><br>
• Parseas el precio strings como "1,600.00" → float 1600.0<br>
• Manejar los datos faltantes (price: 24%, availability: 33%, description: 92%)<br>
• Estandarizar el texto de disponibilidad en categorías limpias<br>
• Eliminar duplicados
</div>

In [9]:
# ============================================================
# SECTION 3 — Data Cleaning
# ============================================================
df = df_raw.copy()

# ── Parse price ─────────────────────────────────────────────
def clean_price(p):
    if pd.isna(p): return np.nan
    p = str(p).replace(',', '').strip()
    m = re.search(r'[\d.]+', p)
    return float(m.group()) if m else np.nan

df['price_clean'] = df['price'].apply(clean_price)

# ── Standardise availability ─────────────────────────────────
def clean_availability(a):
    if pd.isna(a): return 'Unknown'
    a = str(a).strip().lower()
    if 'in stock' in a:          return 'In Stock'
    if 'unavailable' in a:       return 'Unavailable'
    if '1 to 3' in a:            return 'Ships 1-3 weeks'
    if '4 to 5' in a or '6 to' in a or '9 to' in a: return 'Ships 4-14 days'
    if '13 to 14' in a or '2 to 3 weeks' in a:      return 'Ships 2+ weeks'
    if 'only' in a and 'left' in a: return 'Low Stock'
    if '2 to 3 days' in a or '1 to 2 days' in a:    return 'Ships 1-3 days'
    return 'Other'

df['avail_clean'] = df['availability'].apply(clean_availability)

# ── Text length features ─────────────────────────────────────
df['name_length']  = df['name'].fillna('').str.len()
df['name_words']   = df['name'].fillna('').str.split().str.len()
df['has_brand']    = df['brand'].notna().astype(int)
df['has_desc']     = df['description'].notna().astype(int)
df['has_specs']    = df['specifications'].notna().astype(int)

# ── Duplicate check ──────────────────────────────────────────
dupes = df.duplicated(subset='asin').sum()
print(f'✅ Price parsed: {df["price_clean"].notna().sum():,} / {len(df):,} products')
print(f'✅ Duplicate ASINs: {dupes}')
print(f'✅ Availability categories: {df["avail_clean"].value_counts().to_dict()}')
df[['name','price_clean','avail_clean','primary_category','name_length']].head(5)

✅ Price parsed: 1,092 / 1,436 products
✅ Duplicate ASINs: 0
✅ Availability categories: {'In Stock': 607, 'Unknown': 477, 'Ships 1-3 weeks': 121, 'Ships 4-14 days': 115, 'Ships 2+ weeks': 80, 'Other': 20, 'Ships 1-3 days': 16}


,name,price_clean,avail_clean,primary_category,name_length
0,Hands-On Design Patterns with C++: Solve commo...,1600.0,Unknown,Books,118
1,The Blazing Horizon The True Story of Pawnee B...,2399.0,Ships 2+ weeks,Books,74
2,Fast and easy lunch box: A collection of 15eas...,NaN,Unknown,Kindle Store,108
3,European Foreign Policy Scorecard 2014,NaN,Unknown,Kindle Store,38
4,The 13th Hour: Chaos (Volume 2) (The Nick Quin...,1849.0,Ships 2+ weeks,Books,64
